# Import libraries and components

In [1]:
# Import necessary modules
import os
import matplotlib.pyplot as plt
import numpy as np
import imageio.v2 as imageio

from landlab import HexModelGrid, RasterModelGrid
from landlab.components import FastscapeEroder, FlowAccumulator, NormalFault
from landlab.plot import imshow_grid

In [2]:
%matplotlib inline

# Parameters

In [3]:
# Parameters to change
K = 0.0005   # stream power coefficient, bigger = streams erode more quickly
U = 0.01    # uplift rate in meters per year
dt = 1000    # time step in years
dx = 10      # space step in meters
nr = 60      # number of model rows
nc = 100     # number of model columns

# Prepare directory to save the frames

In [4]:
import shutil
frame_dir = "frames"
if os.path.exists(frame_dir):
    shutil.rmtree(frame_dir)
os.makedirs(frame_dir, exist_ok=True)
frame_paths = []

In [5]:
n_steps = 80            # numero di time step totali
frame_every = 1         # salva un frame ogni N step (1 = ogni step)
elev_min, elev_max = 0, 200   # scala colore fissa, adattala ai tuoi valori

# Cartella dove salvare i frame
frame_dir = "frames"
os.makedirs(frame_dir, exist_ok=True)

# Initiate the grid
grid = HexModelGrid((nr, nc), 10, node_layout="rect")

# Add a topographic__elevation field with noise
z = grid.add_zeros("topographic__elevation", at="node")
z[grid.core_nodes] += 100.0 + np.random.randn(grid.core_nodes.size)

fr = FlowAccumulator(grid)
fs = FastscapeEroder(grid, K_sp=K)
nf = NormalFault(
    grid,
    fault_trace={"x1": 0, "x2": 800, "y1": 0, "y2": 500},
    include_boundaries=True,
)

frame_paths = []

def save_frame(step_index, time_years):
    """Salva la topografia corrente come immagine PNG."""
    fig, ax = plt.subplots(figsize=(8, 5))
    imshow_grid(
        grid,
        "topographic__elevation",
        cmap="terrain",
        colorbar_label="Elevation (m)",
        limits=(elev_min, elev_max),
    )
    ax.set_title(f"t = {time_years:,} years")
    fname = os.path.join(frame_dir, f"frame_{step_index:04d}.png")
    fig.savefig(fname, dpi=150)   # <-- niente bbox_inches="tight"
    plt.close(fig)
    frame_paths.append(fname)

# Frame iniziale (t = 0)
save_frame(0, 0)

# Run this model for 20 1000-year timesteps (20_000 years).
for i in range(n_steps):
    fr.run_one_step()
    fs.run_one_step(dt)
    z[grid.core_nodes] += U * dt

    if (i + 1) % frame_every == 0:
        save_frame(i + 1, (i + 1) * dt)

# Earthquake con 50 m displacement (opzionale, decommenta se vuoi includerlo)
nf.run_one_earthquake(dz=100)
for i in range(n_steps):
     fr.run_one_step()
     fs.run_one_step(dt)
     z[grid.core_nodes] += U * dt
     save_frame(n_steps + i + 1, (n_steps + i + 1) * dt)

# --- Assembla i frame in un video MP4 ---
output_video = "topo_evolution.mp4"
fps = 5  # fotogrammi al secondo (regola la velocità del video)

output_gif = "topo_evolution.gif"
with imageio.get_writer(output_gif, mode="I", fps=fps) as writer:
    for path in frame_paths:
        img = imageio.imread(path)
        writer.append_data(img)

print(f"GIF salvata in: {output_gif}")

from IPython.display import Image as IPImage
IPImage(output_gif)

print(f"Video salvato in: {output_video}")

# Per visualizzarlo direttamente nel notebook:
from IPython.display import Video
Video(output_video, embed=True, width=700)

/opt/tljh/user/envs/ivy/lib/python3.13/site-packages/landlab/graph/sort/sort.py:724: UserWarning: 'where' used without 'out', expect uninitialized memory in output. If this is intentional, use out=None.
  angle_of_spoke_at_hub = np.arctan2(dy, dx, where=spokes_at_hub != -1)


GIF salvata in: topo_evolution.gif
Video salvato in: topo_evolution.mp4
